# Chapter 3: Coding Attention Mechanisms

Reference: `LLMs-from-scratch/ch03/`

**Goals:**
- Implement self-attention from scratch
- Implement causal (masked) attention
- Implement multi-head attention
- Understand scaled dot-product attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Self-Attention

In [ ]:
#Simple self attention for a 4-token sequence, each token is represented by a 3-dimensional vector
input=torch.tensor([
    [0.43, 0.14, 0.89],
    [0.55, 0.87, 0.66],
    [0.57, 0.85, 0.64],
    [0.22, 0.58, 0.33],
])

#Step 1: Compute raw attention scores (dot product of input with its transpose)
scores = input @ input.T
print("Raw attention scores:\n", scores)

#Step 2: Apply softmax to get attention weights
attention_weights = torch.softmax(scores, dim=-1)
print("Attention weights:\n", attention_weights)

#Step 3: Compute the weighted sum of the input vectors
output = attention_weights @ input
print("Output of self-attention:\n", output)

Raw attention scores:
 tensor([[0.9966, 0.9457, 0.9337, 0.4695],
        [0.9457, 1.4950, 1.4754, 0.8434],
        [0.9337, 1.4754, 1.4570, 0.8296],
        [0.4695, 0.8434, 0.8296, 0.4937]])
Attention weights:
 tensor([[0.2874, 0.2731, 0.2699, 0.1696],
        [0.1875, 0.3248, 0.3185, 0.1693],
        [0.1884, 0.3239, 0.3180, 0.1698],
        [0.2036, 0.2959, 0.2919, 0.2086]])
Output of self-attention:
 tensor([[0.4649, 0.6056, 0.6647],
        [0.4780, 0.6777, 0.6409],
        [0.4777, 0.6769, 0.6409],
        [0.4626, 0.6550, 0.6322]])


In [ ]:
# Learnable projection matrices
x = input #shape [4, 3]
d_in = 3  # input dimension
d_out = 3  # output dimension

W_q = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))
W_k = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))
W_v = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))

# Compute queries, keys, and values
Q = x @ W_q  # shape [4, 3]
K = x @ W_k  # shape [4, 3]
V = x @ W_v  # shape [4, 3]

# Scaled dot-product attention
scores = Q @ K.T 
Scaled_scores = scores / (d_out ** 0.5)  #scaled to prevent large values
attention_weights = F.softmax(Scaled_scores, dim=-1)
output = attention_weights @ V

print("Q Shape:", Q.shape)
print("K Shape:", K.shape)
print("V Shape:", V.shape)
print("attention_weights", attention_weights)
print("Output", output)

Q Shape: torch.Size([4, 3])
K Shape: torch.Size([4, 3])
V Shape: torch.Size([4, 3])
attention_weights tensor([[0.1999, 0.2875, 0.2756, 0.2370],
        [0.2967, 0.2214, 0.2249, 0.2569],
        [0.2967, 0.2212, 0.2249, 0.2572],
        [0.2876, 0.2269, 0.2302, 0.2553]], grad_fn=<SoftmaxBackward0>)
Output tensor([[ 0.6477,  0.2545, -0.6658],
        [ 0.6565,  0.2256, -0.6350],
        [ 0.6564,  0.2256, -0.6349],
        [ 0.6556,  0.2283, -0.6378]], grad_fn=<MmBackward0>)


## 2. Causal (Masked) Attention

In [ ]:
#Same Q, K, V as before
x = input

d_in = 3  # input dimension
d_out = 3  # output dimension

W_q = torch.randn(d_in, d_out)
W_k = torch.randn(d_in, d_out)
W_v = torch.randn(d_in, d_out)

Q = x @ W_q  # shape [4, 3]
K = x @ W_k  # shape [4, 3]
V = x @ W_v  # shape [4, 3] 

# Compute raw attention scores
scores = Q @ K.T/(d_out ** 0.5)

# Create casual mask : upper triangular matrix with -inf
n = scores.size(0)
mask = torch.triu(torch.ones(n, n) * float('-inf'), diagonal=1)
print("Casual Mask:\n", mask)

# Apply mask to scores
masked_scores = scores + mask
attention_weights = torch.softmax(masked_scores, dim=-1)
output = attention_weights @ V

print("Masked Attention Scores:\n", masked_scores)
print("Attention Weights after Masking:\n", attention_weights)
print("Output of masked self-attention:\n", output)

Casual Mask:
 tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])
Masked Attention Scores:
 tensor([[ 1.0070,    -inf,    -inf,    -inf],
        [-0.3880, -0.4024,    -inf,    -inf],
        [-0.4011, -0.4290, -0.4023,    -inf],
        [-0.3649, -0.4166, -0.4063, -0.2327]])
Attention Weights after Masking:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5036, 0.4964, 0.0000, 0.0000],
        [0.3365, 0.3273, 0.3361, 0.0000],
        [0.2469, 0.2345, 0.2369, 0.2818]])
Output of masked self-attention:
 tensor([[-0.4859,  0.4369,  1.9489],
        [-0.7165,  0.7390,  1.8106],
        [-0.7886,  0.8610,  1.7674],
        [-0.7233,  0.7319,  1.4709]])


## 3. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qvk_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads #dimension of each head

        self.W_query = nn.Linear(d_in, d_out, bias=qvk_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qvk_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qvk_bias)
        self.out_proj = nn.Linear(d_out, d_out) #final output projection
        self.dropout = nn.Dropout(dropout)

        #casual mask (register as buffer so it moves with the model)
        self.register_buffer('mask', mask)
    def forward(self, x):
        b,n,d=x.shape #batch size, sequence length, input dimension

        #1. Linear projections
        Q = self.W_query(x) #shape [b, n, d_out]
        K = self.W_key(x)   #shape [b, n, d_out]
        V = self.W_value(x) #shape [b, n, d_out]

        #2. Reshape for multi-head attention (b,n,d_out) -> (b, num_heads, n, head_dim)
        Q = Q.view(b, n, self.num_heads, self.head_dim).transpose(1, 2) #shape [b, num_heads, n, head_dim]
        K = K.view(b, n, self.num_heads, self.head_dim).transpose(1, 2) #shape [b, num_heads, n, head_dim]
        V = V.view(b, n, self.num_heads, self.head_dim).transpose(1, 2) #shape [b, num_heads, n, head_dim]

        #3. Scaled dot-product attention
        scores = Q @ K.transpose(-2, -1) / (self.head_dim ** 0.5) #shape [b, num_heads, n, n]
        scores += self.mask[:n,:n] #apply casual mask
        attention_weights = F.softmax(scores, dim=-1) #shape [b, num_heads, n, n]
        attention_weights = self.dropout(attention_weights) #apply dropout to attention weights

        #4. Weighted sum of values
        context = attention_weights @ V #shape [b, num_heads, n, head_dim]

        #5. Concatenate heads and project
        context = context.transpose(1, 2).contiguous().view(b, n,self.d_out) #shape [b, n, d_out]
        output = self.out_proj(context) #shape [b, n, d_out]
        return output
    
#Test
batch_size = 2
seq_length = 4
d_in = 3
d_out = 6
num_heads = 2

x = torch.randn(batch_size, seq_length, d_in)
mha = MultiHeadAttention(d_in, d_out, context_length=1024, dropout=0.0, num_heads=num_heads)
output = mha(x)
print(f"Input shape: {x.shape} (batch_size={batch_size}, seq_length={seq_length}, d_in={d_in})")
print(f"Output shape: {output.shape} (batch_size={batch_size}, seq_length={seq_length}, d_out={d_out}))")


Input shape: torch.Size([2, 4, 3]) (batch_size=2, seq_length=4, d_in=3)
Output shape: torch.Size([2, 4, 6]) (batch_size=2, seq_length=4, d_out=6))
